In [1]:
import numpy as np
import pandas as pd

import re

from libraries.preprocess import preprocess


df = pd.read_csv("data/extracted/extraction_2024-08-12_2_spain.csv")

df.head(3)

,title,company_name,location,via,share_link,extensions,description,apply_options,job_id,posted_at,schedule_type,thumbnail,country_search,work_from_home,salary,date_posted
0,Consultor tecnológico,Atos,"Madrid, Spain",Jobs At Atos,https://www.google.com/search?ibp=htl;jobs&q=C...,"['8 days ago', 'Full-time']",Eviden es una línea de negocio del Grupo Atos ...,"[{'title': 'Jobs At Atos', 'link': 'https://jo...",eyJqb2JfdGl0bGUiOiJDb25zdWx0b3IgdGVjbm9sw7NnaW...,8 days ago,Full-time,NaN,Spain,NaN,NaN,2024-08-12
1,Consultor/a funcional SAP SD/MM,DESIC | Consultora Tecnológica,Spain,LinkedIn,https://www.google.com/search?ibp=htl;jobs&q=C...,"['13 hours ago', 'Full-time', 'No degree menti...",En DESIC Trabajamos en el desarrollo de soluci...,"[{'title': 'LinkedIn', 'link': 'https://es.lin...",eyJqb2JfdGl0bGUiOiJDb25zdWx0b3IvYSBmdW5jaW9uYW...,13 hours ago,Full-time,NaN,Spain,NaN,NaN,2024-08-12
2,Consultor Digital - MutuaCity,Mutua Madrileña,"Community of Madrid, Spain",JobLeads,https://www.google.com/search?ibp=htl;jobs&q=C...,"['21 hours ago', 'Full-time', 'No degree menti...",¡Únete al equipo que lidera el mejor Patrimoni...,"[{'title': 'JobLeads', 'link': 'https://www.jo...",eyJqb2JfdGl0bGUiOiJDb25zdWx0b3IgRGlnaXRhbCAtIE...,21 hours ago,Full-time,NaN,Spain,NaN,NaN,2024-08-12


In [2]:
skills = pd.read_csv("data/Competencias Técnicas-UPDATE.csv")["Competencia"]

def get_skills(string, skills = skills):
    
    tech_skills = list()
    
    for skill in skills:
        try:
            if len(skill) > 3:
                if (skill.lower() in string.lower().split()) or (skill.lower() in string.lower()):
                    tech_skills.append(skill)
            else:
                if skill.lower() in string.lower().split():
                    tech_skills.append(skill)
        except:
            tech_skills.append(np.nan)
    
    return tech_skills

In [3]:
# Eliminamos duplicados
df = df.drop_duplicates("job_id").reset_index(drop = True) 

# Limpiamos "via"
for i in range(len(df["via"])):
    
    # Debido a valores NoneType, usamos try/except
    
    try:
        df.loc[i, "via"] = preprocess.clean_source(df.loc[i, "via"]) # Cada fila
        
    except:
        df.loc[i, "via"] = np.nan
    
# Limpiamos "location"
df["location"] = df["location"].apply(lambda x : preprocess.clean_location(x))

# Limpiamos "contract_type"
df["schedule_type"] = df["schedule_type"].apply(lambda x : preprocess.clean_contract_type(x))

# Limpiamos "created_date"
df["posted_at"] = df["posted_at"].apply(lambda x : preprocess.transform_date(x))

df["date_posted"] = [preprocess.get_date(x, y) for x, y in df[["date_posted", "posted_at"]].values]

# Creamos "tech_skills"

tech_skills = list()
for i in range(len(df["description"])):
    
    # Debido a valores NoneType, usamos try/except
    
    try:
        tech_skills.append(get_skills(df.loc[i, "description"])) # Cada fila
        
    except:
        tech_skills.append(np.nan)

df["tech_skills"] = tech_skills

In [4]:
# Perfil y especialidades de los trabajos
df_perfil_especialidad = pd.read_csv("data/StackMinimo - StackTec.csv", skiprows = 2).drop("Unnamed: 0", axis = 1)

list_especialidad = df_perfil_especialidad["N2 Especialidad"].dropna().to_list()

list_especialidad = ["Backend",
                     "Base de datos",
                     "Bases de datos",
                     "APIs",
                     "Mobile",
                     "Frontend Movil",
                     "Frontend Web",
                     "Full Stack",
                     "Integración",
                     "Liferay",
                     "Power BI",
                     "Software Release Engineer",
                     "SRE",
                     "Diseñador Gráfico",
                     "Graphic Designer",
                     "Infraestructura cloud",
                     "Estructuras cloud",
                     "Sistemas Operativos",
                     "Seguridad",
                     "Mantenimiento y Soporte",
                     "Administrador de Base de datos",
                     "Servidores y aplicaciones",
                     "Tester funcional",
                     "Tester automatizada",
                     "SCRUM Master",
                     "Product Owner",
                     "Customer Success",
                     "Gerente de Proyecto",
                     "Analisis de Datos",
                     "Analista de Datos"]

dict_perfiles = {v : k for k, v in df_perfil_especialidad[["N1 Perfil ", "N2 Especialidad"]].dropna().values}

# Corrección de algunas keys
dict_perfiles = {"Backend": "Desarrollador",
                 "Base de datos": "Desarrollador",
                 "APIs": "Desarrollador",
                 "Mobile": "Desarrollador",
                 "Frontend Movil": "Desarrollador",
                 "Frontend Web": "Desarrollador",
                 "Full Stack": "Desarrollador",
                 "Integración": "Desarrollador",
                 "Liferay": "Desarrollador",
                 "Power BI": "Desarrollador",
                 "Software Release Engineer": "Devops",
                 "Diseñador Gráfico": "Diseñador",
                 "Infraestructura cloud": "Infraestructura",
                 "Sistemas Operativos": "Infraestructura",
                 "Seguridad": "Infraestructura",
                 "Mantenimiento y Soporte": "Infraestructura",
                 "Tester funcional": "Quality Assurance",
                 "Tester automatizada": "Quality Assurance",
                 "SCRUM Master": "Gestión Operativa",
                 "Product Owner": "Gestión Operativa",
                 "Customer Success": "Gestión Operativa",
                 "Gerente de Proyecto": "Gestión Operativa",
                 "Analisis de Datos": "Especialista"}

# Funciones para generar las columnas de perfil y especialidad
def get_especialidad(string, list_especialidad):
    
    especialidades = list({especialidad for especialidad in list_especialidad if especialidad.lower() in string.lower()})
    
    return especialidades if especialidades else np.nan

def get_perfil(lista, dict_perfiles):
    
    perfiles = list({v for k, v in dict_perfiles.items() if k in lista})
    
    return perfiles if perfiles else np.nan

# Funciones para generar las columnas de años de experiencia y nivel de experiencia
def find_years_of_experience(string: str):
    
    list_strings = ["años de", "years of", "years experience", "años experiencia"]
    
    string = string.lower()
    
    years = [string[string.find(s) - 5 : string.find(s) + len(s) + 1] for s in list_strings if string.find(s) != -1]
    
    numeros = [re.findall(r"\d+", y) for y in years]

    numeros = [[int(n) for n in num if 0 < int(n) < 13] for num in numeros]
    
    numeros = [max(num) if num else np.nan for num in numeros]

    return max(numeros) if numeros else np.nan


def experience_level(num):
    
    if not pd.isna(num):
    
        if num < 2:
            return "Junior"
        elif num <= 4:
            return "Semi-Senior"
        elif num < 8:
            return "Senior"
        else:
            return "Leader"
        
    else:
        return np.nan

In [5]:
df["job_specialization"] = df["description"].apply(lambda x : get_especialidad(x, list_especialidad = list_especialidad) if not pd.isna(x) else None)
df["job_profile"] = df["description"].apply(lambda x : get_perfil(x, dict_perfiles = dict_perfiles) if not pd.isna(x) else None)

df["experience"] = df["description"].apply(lambda x : find_years_of_experience(x) if not pd.isna(x) else x)
df["experience_level"] = df["experience"].apply(lambda x : experience_level(x))

In [6]:
# Actualizamos "contract_type"
df["schedule_type"] = df["schedule_type"]\
                            .apply(lambda x : "Full-time" if x == "Tiempo completo" else x)

# Creamos "remote_work"
def get_remote_work(string):
    
    resultados = list()
    
    if "remoto" in string or "remote work" in string or "remote" in string:
        
        resultados.append("Remoto")
        
    elif "hibrido" in string or "hybrid" in string or "híbrido" in string:
        
        resultados.append("Hibrido")
        
    elif "presencial" in string or "in-office" in string:
        
        resultados.append("Presencial")
        
    else:
        return np.nan
        
    return resultados

df["remote_work"] = df["description"].apply(lambda x : get_remote_work(x))

In [7]:
df = df[["job_id", "experience", "experience_level", "description", "job_specialization",
                     "job_profile", "remote_work", "tech_skills", "title", "company_name", "location", "via",
                     "date_posted", "schedule_type", "share_link"]]

In [8]:
df.columns

Index(['job_id', 'experience', 'experience_level', 'description',
       'job_specialization', 'job_profile', 'remote_work', 'tech_skills',
       'title', 'company_name', 'location', 'via', 'date_posted',
       'schedule_type', 'share_link'],
      dtype='object')

In [9]:
df = df.replace({np.nan : None})

df["remote_work"] = df["remote_work"].apply(lambda x : x[0] if isinstance(x, list) else x)


In [10]:
df.to_csv("spain_2024_08_12_main.csv", index = False)

In [11]:
tabla_specialization = df[["job_id", "job_specialization"]].explode("job_specialization").dropna()

tabla_specialization.to_csv("spain_2024_08_12_especializacion.csv", index = False) 

In [12]:
tabla_profile = df[["job_id", "job_profile"]].explode("job_profile").dropna()

tabla_profile.to_csv("spain_2024_08_12_profile.csv", index = False) 

In [13]:
tabla_tech_skills = df[["job_id", "tech_skills"]].explode("tech_skills").dropna()

tabla_tech_skills.to_csv("spain_2024_08_12_skills.csv", index = False)

In [14]:
df.columns = ['job_id', 'experience', 'experience_level', 'description',
       'job_specialization', 'job_profile', 'remote_work', 'tech_skills',
       'title', 'company_name', 'location', 'source', 'date_posted',
       'contract_type', "url"]

In [15]:
df_skills = pd.read_csv("data/Competencias Técnicas-UPDATE.csv")
df_skills = df_skills[["Competencia"]].reset_index()

df_skills.columns = ["competencia_id", "Competencia"]

df_skills["competencia_id"] = df_skills["competencia_id"] + 1

df_skills.to_csv("tabla_skills.csv", index = False)

df_skills

,competencia_id,Competencia
0,1,.NET
1,2,.NETCore
2,3,Abstract Factory
3,4,Aceptación
4,5,Acknowledgement
...,...,...
1234,1235,YouTube
1235,1236,Zbrush
1236,1237,Zen Cart
1237,1238,Zendesk


In [16]:
# Tabla Tech Skills
tabla_tech_skills = tabla_tech_skills.merge(df_skills, left_on = "tech_skills", right_on = "Competencia")[["job_id", "competencia_id"]]

tabla_tech_skills.to_csv("spain_2024_08_12_skills.csv", index = False)

In [17]:
df_perfil = pd.read_csv("data/Tabla Maestra de Etiquetas - Etiquetas Perfil - Especialidad.csv")
tabla_profile = tabla_profile.merge(df_perfil, left_on = "job_profile", right_on = "N1 Perfil")[["job_id", "perfil_id"]]
tabla_profile.to_csv("spain_2024_08_12_profile.csv", index = False)

KeyError: "['perfil_id'] not in index"

In [ ]:
tabla_specialization

In [ ]:
tabla_specialization = tabla_specialization.merge(df_especialidad, left_on = "job_specialization", right_on = "N2 Especialidad")[["job_id", "especialidad_id"]]
tabla_specialization.to_csv("spain_2024_08_12_especializacion.csv", index = False)

In [ ]:
tabla_specialization

In [ ]:
df_perfil = pd.read_csv("data/Tabla Maestra de Etiquetas - Etiquetas Perfil - Especialidad.csv")
df_perfil = df_perfil.drop_duplicates("N1 Perfil")
df_perfil = df_perfil[["N1 Perfil"]].reset_index(drop = True).reset_index()

df_perfil.columns = ["perfil_id", "N1 Perfil"]

df_perfil["perfil_id"] = df_perfil["perfil_id"] + 1

df_perfil.to_csv("tabla_perfil.csv", index = False)

df_perfil

In [ ]:
df_especialidad = pd.read_csv("data/Tabla Maestra de Etiquetas - Etiquetas Perfil - Especialidad.csv")

df_especialidad = df_especialidad[["N2 Especialidad"]].drop_duplicates()
df_especialidad = df_especialidad.reset_index(drop = True).reset_index()

df_especialidad.columns = ["especialidad_id", "N2 Especialidad"]

df_especialidad["especialidad_id"] = df_especialidad["especialidad_id"] + 1

df_especialidad.to_csv("tabla_especialidad.csv", index = False)
df_especialidad

In [23]:
df.columns = ['job_id', 'experience', 'experience_level', 'description',
       'Especialidad', 'Perfil', 'remote_work', 'tech_skills',
       'title', 'company_name', 'location', 'source', 'date_posted',
       'contract_type', "url"]

In [24]:
from libraries.preprocess import airtable

In [25]:
airtable_key = "pat95dOl3maQP62MP.a7c882a9f4e60ee5f04509619bd08fa3560a121aef9fa0b5a07131fb069fe69e"
base = "appU0CLDju62P5dKe"
table = "tbltjuRtmyKZsuBZQ"

airtable.airtable_post_spain(df = df, airtable_key = airtable_key, base = base, table = table)

200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200
200


In [ ]:
##############################################################################################################################